## 0. Install Requirements

In [ ]:
!pip install -q "transformers==4.44.2" "datasets" "evaluate" "rouge_score" "accelerate" torch sentencepiece


## 1. Upload the Dataset

Upload `Short_Story_Dataset.csv` (required columns: `Story Content` and `Abstract`).


In [ ]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]
print("Uploaded file:", csv_path)


## 2. Load & Inspect the Data

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
print("Number of rows:", len(df))
print("Columns:", df.columns.tolist())
df.head(3)


In [ ]:
# Word-count stats (used later to pick sensible max_length values)
df["story_word_count"] = df["Story Content"].astype(str).apply(lambda x: len(x.split()))
df["abstract_word_count"] = df["Abstract"].astype(str).apply(lambda x: len(x.split()))

print(df[["story_word_count", "abstract_word_count"]].describe())


In [ ]:
import re
import unicodedata

def clean_text(text: str) -> str:
    """Thorough text-cleaning pass for story content and abstracts."""
    text = str(text)

    # 1. Unicode normalization (fixes weird composed/decomposed accented chars)
    text = unicodedata.normalize("NFKC", text)

    # 2. Normalize smart/curly quotes and dashes to plain ASCII equivalents
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-").replace("\u2011", "-")

    # 3. Normalize ellipsis character to three dots
    text = text.replace("\u2026", "...")

    # 4. Replace non-breaking / narrow no-break spaces with a regular space
    text = text.replace("\xa0", " ").replace("\u202f", " ")

    # 5. Remove control characters (except normal whitespace)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)

    # 6. Remove any leftover non-Latin script noise (keep basic Latin + accented Latin + punctuation)
    text = re.sub(r"[^\x00-\x7F\u00C0-\u024F]", "", text)

    # 7. Collapse newlines/tabs into single spaces (BART treats stories as flat text)
    text = re.sub(r"[\r\n\t]+", " ", text)

    # 8. Collapse multiple spaces into one
    text = re.sub(r" {2,}", " ", text)

    # 9. Collapse repeated punctuation (e.g. "!!!" -> "!", "??" -> "?", "...." -> "...")
    text = re.sub(r"([!?])\1{1,}", r"\1", text)
    text = re.sub(r"\.{4,}", "...", text)

    # 10. Trim stray spaces before punctuation (e.g. "word ." -> "word.")
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    return text.strip()


# Quick before/after sanity check on one row
sample_raw = df.loc[df["Story Content"].astype(str).str.contains("\n"), "Story Content"].iloc[0]
print("BEFORE:")
print(repr(sample_raw[:200]))
print()
print("AFTER:")
print(repr(clean_text(sample_raw)[:200]))


In [ ]:
df["Story Content"] = df["Story Content"].apply(clean_text)
df["Abstract"] = df["Abstract"].apply(clean_text)

# Recompute word counts after cleaning
df["story_word_count"] = df["Story Content"].apply(lambda x: len(x.split()))
df["abstract_word_count"] = df["Abstract"].apply(lambda x: len(x.split()))

print("Cleaning applied to Story Content and Abstract.")
df[["story_word_count", "abstract_word_count"]].describe()


In [ ]:
# Drop duplicate stories, empty rows, and very short/corrupted stories
MIN_STORY_WORDS = 20

before = len(df)
df = df.drop_duplicates(subset=["Story Content"])
df = df[df["Story Content"].str.strip() != ""]
df = df[df["Abstract"].str.strip() != ""]
df = df[df["story_word_count"] >= MIN_STORY_WORDS]
df = df.reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} duplicate/empty/too-short rows.")
print(f"Remaining rows: {after}")


## 4. Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

data = df[["Story Content", "Abstract"]].rename(
    columns={"Story Content": "text", "Abstract": "summary"}
)

train_df, temp_df = train_test_split(data, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


In [ ]:
from datasets import Dataset, DatasetDict

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

raw_datasets


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "sshleifer/distilbart-cnn-12-6"  # switch to "facebook/bart-large-cnn" for a stronger GPU

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Device:", device)


In [ ]:
MAX_INPUT_LENGTH = 512    # most stories are longer; they will be truncated
MAX_TARGET_LENGTH = 150   # covers most abstracts (75th percentile is ~131 words)

def preprocess_function(examples):
    inputs = examples["text"]
    targets = examples["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

tokenized_datasets


## 7. Evaluation Metric (ROUGE)

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    result = {k: round(v * 100, 2) for k, v in result.items()}

    pred_lens = [len(tokenizer(pred)["input_ids"]) for pred in decoded_preds]
    result["gen_len"] = round(np.mean(pred_lens), 2)

    return result


## 8. Baseline Evaluation (Before Fine-tuning)

We measure the off-the-shelf model's performance on the test set first, so we can compare it
against the fine-tuned model later.


In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

baseline_args = Seq2SeqTrainingArguments(
    output_dir="./baseline_eval",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    report_to="none",
)

baseline_trainer = Seq2SeqTrainer(
    model=model,
    args=baseline_args,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

baseline_metrics = baseline_trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print("Baseline results (before fine-tuning):")
baseline_metrics


## 9. Fine-tuning

Now we actually train the model on our story dataset. Adjust `num_train_epochs` and
`per_device_train_batch_size` based on your GPU's speed/memory. If you hit a
`CUDA out of memory` error, lower `per_device_train_batch_size` to 2 and/or raise
`gradient_accumulation_steps`.


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./story-summarizer-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=4,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


In [ ]:
train_result = trainer.train()
train_result


## 10. Evaluate the Fine-tuned Model

In [ ]:
finetuned_metrics = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print("Results after fine-tuning:")
finetuned_metrics


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["rouge1", "rouge2", "rougeL", "rougeLsum", "gen_len"],
    "Before Fine-tuning": [
        baseline_metrics.get("eval_rouge1"),
        baseline_metrics.get("eval_rouge2"),
        baseline_metrics.get("eval_rougeL"),
        baseline_metrics.get("eval_rougeLsum"),
        baseline_metrics.get("eval_gen_len"),
    ],
    "After Fine-tuning": [
        finetuned_metrics.get("eval_rouge1"),
        finetuned_metrics.get("eval_rouge2"),
        finetuned_metrics.get("eval_rougeL"),
        finetuned_metrics.get("eval_rougeLsum"),
        finetuned_metrics.get("eval_gen_len"),
    ],
})
comparison


## 11. Save the Fine-tuned Model

In [ ]:
SAVE_DIR = "./story-summarizer-final"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")


In [ ]:
# Optional: zip the folder and download it to your machine
import shutil
shutil.make_archive("story-summarizer-final", "zip", SAVE_DIR)

from google.colab import files
files.download("story-summarizer-final.zip")


## 12. Try It Out

Test the fine-tuned model on a story from the dataset, or write your own.


In [ ]:
def summarize_story(text, model, tokenizer, max_length=150, min_length=20):
    text = clean_text(text)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# Example: take a story from the test set and compare the generated summary to the ground truth
sample = test_df.iloc[0]
generated_summary = summarize_story(sample["text"], model, tokenizer)

print("Original text (first 400 characters):")
print(sample["text"][:400] + "...")
print()
print("Ground-truth summary:")
print(sample["summary"])
print()
print("Model-generated summary (after fine-tuning):")
print(generated_summary)
